# Plan B Colab : run d'ablation d'un mélange (M1 à M5)

À utiliser quand Kaggle est bloqué. Même protocole que `kaggle/mix-template/script.py` : kit `3709c85`, config figée (graine 1337, 3 052 pas), même tokenizer, `--micro-batch 8`. Seules les données changent.

1. Exécution → Modifier le type d'exécution → **GPU T4**.
2. Dans le panneau 🔑 *Secrets* de Colab, ajoute `KAGGLE_API_TOKEN` (ton jeton Kaggle) et active l'accès pour ce notebook. **Ne colle jamais le jeton dans une cellule.**
3. Choisis le mélange ci-dessous, puis Exécution → **Tout exécuter**. Garde l'onglet ouvert (~15-20 min).
4. À la fin, `evaluation.json` se télécharge : donne-le à l'orchestrateur.

In [ ]:
# Mélange à entraîner : M3, M1, M2, M4 ou M5 (le dataset Kaggle privé piscine-mix-<id> doit exister).
MIX = "M1"

In [ ]:
# Jeton Kaggle lu depuis les secrets Colab (jamais écrit dans le notebook).
import os
from google.colab import userdata
os.environ["KAGGLE_API_TOKEN"] = userdata.get("KAGGLE_API_TOKEN")
# Vérifie qu'un GPU est bien attribué.
!nvidia-smi --query-gpu=name --format=csv

In [ ]:
# Télécharge le flux du mélange (train.bin + data_meta.json) et le tokenizer commun.
!pip install -q kaggle tokenizers==0.23.3
!kaggle datasets download matissegarlot/piscine-mix-{MIX.lower()} -p /content/data --unzip
from pathlib import Path
data = next(Path("/content/data").rglob(f"{MIX}/train.bin")).parent
tokenizer = next(Path("/content/data").rglob("tokenizer.json"))
print("données :", data, "| tokenizer :", tokenizer)

In [ ]:
# Même version du kit que pour toutes les mesures de l'ablation.
!git clone -q https://github.com/Meuleur/Piscine.git /content/piscine-kit
!cd /content/piscine-kit && git checkout -q --detach 3709c85ddaabfcd688d6eea80a5c5ed90605b549 && git log -1 --oneline

In [ ]:
# Entraînement officiel (config et hyperparamètres figés par le kit), puis BPB sur les 230 documents publics.
out = f"/content/run-{MIX}"
!cd /content/piscine-kit && python -m ftb.train --data {data} --tokenizer {tokenizer} --out {out} --device cuda:0 --micro-batch 8
!cd /content/piscine-kit && python -m ftb.evaluate --ckpt {out}/ckpt.pt --tokenizer {tokenizer} --data data/validation.jsonl --device cuda:0 --out {out}/evaluation.json

In [ ]:
# Affiche le résultat et télécharge les petits fichiers (pas le checkpoint).
import shutil
shutil.copyfile(data / "data_meta.json", f"{out}/data_meta.json")
print(open(f"{out}/evaluation.json", encoding="utf-8").read())
from google.colab import files
for name in ("evaluation.json", "data_meta.json", "train_log.jsonl"):
    files.download(f"{out}/{name}")